# Hypoxia signature model — single-dataset discovery + scorer\n\nBuilt incrementally per `CLAUDE.md`. Scope for this session: seed list → discovery (Eqs 1, 2, 5 + Monte Carlo) → scorer (Eq 7 + HS). Adapters, Cox validation, and sigQC are out of scope here."

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## Seed genes (Buffa 2010, HNSCC training network, set A)

The ten seed genes exactly as printed in the paper, hardcoded — not a file to
load. One of the ten (`AK3L1`) was later renamed by HGNC to `AK4`; modern
datasets use the new symbol, so we keep both forms and resolve at lookup time
against whatever gene index the actual dataset has.

In [ ]:
# Ten seed genes, literal names as printed in Buffa et al. 2010.
SEED_GENES = [
    "ADM", "AK3L1", "BNIP3", "CA9", "ENO1",
    "HK2", "LDHA", "PGK1", "SLC2A1", "VEGFA",
]

# AK3L1 -> AK4 was a formal HGNC rename, not a casual alias. Exact-string
# matching against a modern dataset (Ensembl/GEO/TCGA-annotated) will miss
# AK3L1 entirely and silently drop that seed unless we also try AK4.
SEED_ALIASES = {
    "AK3L1": "AK4",
}


def resolve_seed_genes(seed_genes: list[str], available_genes) -> dict[str, str]:
    """Map each seed (paper name) to whichever symbol is present in `available_genes`.

    Tries the literal paper name first, then its known alias. Seeds matching
    neither are left out of the returned dict -- callers should check for
    missing seeds rather than assume all ten resolve.
    """
    available = set(available_genes)
    resolved = {}
    for seed in seed_genes:
        if seed in available:
            resolved[seed] = seed
        elif seed in SEED_ALIASES and SEED_ALIASES[seed] in available:
            resolved[seed] = SEED_ALIASES[seed]
    return resolved

## Discovery step 1: seed–gene affinity + membership (Eqs 1, 2 — exact formulas)

Replacing the reconstructed version from before with the paper's actual
equations, now that we have them.

**Eq 1 — affinity, d(p_i, y_j):**

d(p_i, y_j) = [1 + exp(-(r²(p_i,y_j) - γ_t) / γ_s)]^-1

A sigmoid soft-threshold on `r²` (squared Spearman correlation between seed
`p_i` and gene `y_j`). `γ_t` is the cluster boundary — a Bonferroni-corrected
(α = 0.05) significance threshold on `r²` — and `γ_s` controls the sigmoid's
sharpness. As `γ_s → 0` (the form the paper actually uses in this study),
this collapses to a hard step: `d = 1` if `r² > γ_t`, else `0`. Implemented
below with `gamma_s=0` as the default (hard step); the literal sigmoid is
available by passing a small positive `gamma_s`.

This also resolves the ambiguity flagged in the previous version of this
chunk: `γ_t` is explicitly a threshold *on `r²`*, not on `|r|`. So
`critical_r_squared()` below returns the critical value already squared —
no more guessing which scale it's on.

**Eq 2 — membership, g(y_i, p_k):**

g(y_i, p_k) = d(y_i, p_k) / Σ_{j=1}^{K} d(y_i, p_j)

This is genuinely different from the previous chunk's `gamma = delta *
|rho|`, which was a guess at "increases with `|ρ|`" — wrong, now that the
real formula is available. Eq 2 normalizes gene `y_i`'s affinity to seed
`p_k` against its affinity to *all* `K` seeds combined, so it needs every
seed's affinity for a gene computed together, not one seed handled in
isolation. That's why the functions below build a genes × seeds affinity
matrix first, then normalize each gene's row across seeds.

In [ ]:
def critical_r_squared(n: int, m: int, alpha: float = 0.05) -> float:
    """gamma_t in Eq 1: critical r^2 for significance at `alpha`,
    Bonferroni-corrected across `m` comparisons, for a sample of size `n`.

    Finds the critical t-statistic (two-tailed, df = n - 2) for the
    corrected alpha, converts it to a critical |r| via the standard
    r <-> t relationship (t = r * sqrt((n - 2) / (1 - r^2))), then squares
    it, since Eq 1 thresholds r^2 directly.
    """
    alpha_corrected = alpha / m
    df = n - 2
    t_crit = stats.t.ppf(1 - alpha_corrected / 2, df)
    r_crit = t_crit / np.sqrt(df + t_crit ** 2)
    return r_crit ** 2


def seed_gene_correlations(matrix: pd.DataFrame, seed_symbols: list[str]) -> pd.DataFrame:
    """Spearman rho between every gene (column of `matrix`) and each seed gene.

    matrix       : samples x genes matrix
    seed_symbols : gene symbols to correlate against, as they appear in
                   matrix.columns (i.e. resolve_seed_genes(...).values())

    Returns a genes x seeds DataFrame of rho.
    """
    return pd.DataFrame({
        seed: matrix.corrwith(matrix[seed], method="spearman")
        for seed in seed_symbols
    })


def affinity(rho: pd.DataFrame, gamma_t: float, gamma_s: float = 0.0) -> pd.DataFrame:
    """Eq 1: d(p_i, y_j) -- seed-gene affinity, genes x seeds.

    rho     : genes x seeds Spearman correlations (seed_gene_correlations())
    gamma_t : critical r^2 threshold (critical_r_squared())
    gamma_s : sigmoid sharpness. 0 (default) -> hard step, the gamma_s -> 0
              limit the paper uses in this study. A small positive value
              gives the literal sigmoid instead.
    """
    r2 = rho ** 2
    if gamma_s == 0:
        return (r2 > gamma_t).astype(float)
    return 1.0 / (1.0 + np.exp(-(r2 - gamma_t) / gamma_s))


def membership(d: pd.DataFrame) -> pd.DataFrame:
    """Eq 2: g(y_i, p_k) = d(y_i, p_k) / sum_j d(y_i, p_j).

    Normalizes each gene's row of affinities across all K seeds. Genes with
    zero affinity to every seed (the common case) hit 0/0 -- defined here as
    0 for all seeds, not NaN.
    """
    row_sums = d.sum(axis=1)
    g = d.div(row_sums, axis=0)
    return g.fillna(0.0)